# ClearML PlantVillage Workshop — All-in-One  🌱

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kartik-nighania/clearml_tutorial/blob/main/workshop/workshop_full.ipynb)

One notebook, the whole 4-hour workshop. **Run it top to bottom.** Everything shows up live in the ClearML Web UI at **http://202.131.110.56**.

| Hour | What | Runs where |
|---|---|---|
| 1 | Login + first experiment + EDA | here in Colab (CPU) |
| 2 | Fine-tune ViT-Tiny | **submitted to the GPU queue** → the H200 agent |
| 3 | Compare runs, pick the winner | here in Colab |
| 4 | Publish the winner + run inference | here in Colab |

> You only need a browser + a Google account. GPU training runs on the server, not in Colab.

## 0 · Setup — run once per session

Colab resets between sessions, so re-run this if you reconnect. It clones the workshop repo (for the Hour-2/4 scripts) and installs the two libraries Colab is missing (`clearml`, `transformers`; plus `python-dotenv`). Colab already has torch/pandas/matplotlib/etc.

In [ ]:
import os
# Clone the workshop code (idempotent — safe to re-run).
if not os.path.isdir('clearml_tutorial'):
    !git clone -q https://github.com/kartik-nighania/clearml_tutorial.git
if os.path.basename(os.getcwd()) != 'clearml_tutorial':
    os.chdir('clearml_tutorial')
!pip install -q clearml transformers python-dotenv
print('setup ok · cwd =', os.getcwd())

## 1 · Credentials — paste the shared key + set your name

Everyone uses the **same** API key (from the facilitator). You tell your work apart with **`WORKSHOP_USER`** = *your name* → your runs land in your own project `PlantVillage Workshop/<your-name>`.

> We set them with plain `os.environ` (not `%env`) for two reasons: an empty `''` is *really* empty (so the checks below catch it), and these values are **inherited by the `!python` commands** in Hours 2 & 4 — that's the env binding that makes the scripts authenticate.

In [ ]:
import os
# ---- SHARED credentials (same for everyone) — paste between the quotes ----
os.environ['CLEARML_WEB_HOST']       = 'http://202.131.110.56'
os.environ['CLEARML_API_HOST']       = 'http://202.131.110.56:8008'
os.environ['CLEARML_FILES_HOST']     = 'http://202.131.110.56:8081'
os.environ['CLEARML_API_ACCESS_KEY'] = ''   # <-- paste the SHARED access key
os.environ['CLEARML_API_SECRET_KEY'] = ''   # <-- paste the SHARED secret key
os.environ['WORKSHOP_USER']          = ''   # <-- YOUR name, e.g. 'alice' (letters/dashes)

# normalize (strip stray quotes/space so '""' counts as empty) + optional .env fallback + check
for _k in ('CLEARML_API_ACCESS_KEY','CLEARML_API_SECRET_KEY','WORKSHOP_USER',
           'CLEARML_WEB_HOST','CLEARML_API_HOST','CLEARML_FILES_HOST'):
    _v = os.environ.get(_k)
    if _v is not None: os.environ[_k] = _v.strip().strip('"').strip("'").strip()
if not (os.environ.get('CLEARML_API_ACCESS_KEY') and os.environ.get('WORKSHOP_USER')):
    try:
        from dotenv import load_dotenv, find_dotenv; load_dotenv(find_dotenv(usecwd=True), override=True)
    except Exception: pass
assert os.environ.get('CLEARML_API_ACCESS_KEY') and os.environ.get('CLEARML_API_SECRET_KEY'), \
    'API keys are EMPTY — paste the shared key above and re-run.'
assert os.environ.get('WORKSHOP_USER'), 'Set WORKSHOP_USER (your name) above.'
WORKSHOP_USER = os.environ['WORKSHOP_USER']
PROJECT = f'PlantVillage Workshop/{WORKSHOP_USER}'
print('you are:', WORKSHOP_USER, '| your project:', PROJECT)

## 2 · Hour 1 — your first experiment + EDA

`Task.init()` opens a tracked experiment (auto-captures code, packages, env, plots). We do quick EDA on the shared dataset; the bar chart lands in **PLOTS**, the sample-leaf grid in **DEBUG SAMPLES**.

In [ ]:
from clearml import Task, Dataset
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns, random
from PIL import Image

Task.force_store_standalone_script()   # copy the code into the task (no git needed by the agent)
task = Task.init(project_name=PROJECT, task_name='EDA')
print('task page:', task.get_output_log_web_page())

# class distribution from the shared dataset (metadata only, no image download)
try:
    ds = Dataset.get(dataset_project='PlantVillage', dataset_name='plantdisease_full', alias='plantdisease_full')
except Exception:
    ds = Dataset.get(dataset_project='PlantVillage', dataset_name='plantdisease_demo', alias='plantdisease_demo')
files = ds.list_files()
counts = pd.Series([f.split('/')[0] for f in files]).value_counts().sort_values(ascending=False)
print(f'{len(files)} images across {counts.size} classes')
plt.figure(figsize=(11,5)); sns.barplot(x=counts.values, y=counts.index, color='#4C9A2A')
plt.title('PlantVillage — images per class'); plt.tight_layout(); plt.show()
task.get_logger().report_single_value('total_images', len(files))

In [ ]:
# sample-leaf grid -> reported as a PNG so it renders in DEBUG SAMPLES (imshow can't be a Plotly plot)
demo = Dataset.get(dataset_project='PlantVillage', dataset_name='plantdisease_demo', alias='plantdisease_demo').get_local_copy()
demo_classes = sorted(d for d in os.listdir(demo) if os.path.isdir(os.path.join(demo, d)))
random.seed(0)
fig = plt.figure(figsize=(12, 3*len(demo_classes)))
for r, cls in enumerate(demo_classes):
    for c, name in enumerate(random.sample(os.listdir(os.path.join(demo, cls)), 4)):
        ax = fig.add_subplot(len(demo_classes), 4, r*4 + c + 1)
        ax.imshow(Image.open(os.path.join(demo, cls, name)).convert('RGB')); ax.axis('off'); ax.set_title(cls, fontsize=8)
plt.tight_layout()
task.get_logger().report_matplotlib_figure('Sample leaves', 'per class', iteration=0, figure=fig, report_image=True)
plt.show()
task.close()   # end the EDA task before moving on
print('Hour 1 done.')

## 3 · Hour 2 — GPU training on the shared queue

This runs the **`hour2_finetune_vit.py`** script. It configures the task locally, then `execute_remotely('gpu-18gb')` ships it to the GPU queue — a `clearml-agent` on an H200 MIG slice runs the training. It registers one clean model per epoch.

> **Why a `.py` and not cells?** The agent re-runs the submitted code. A plain script is reliable; a notebook would carry `!pip`/`os.environ` magics that break when re-run as a script. The script calls `Task.force_store_standalone_script()`, so ClearML **copies the code into the task** and the agent runs it **without cloning any repo**. Credentials come from the agent's own config on the server (your Colab key was only used to *submit*).

In [ ]:
# submits to the gpu-18gb queue and returns immediately (training continues on the agent).
!python workshop/hour2_finetune_vit.py

## 4 · Wait for training to finish

Hour 2 runs on the agent (~1–2 min). This polls until your latest run completes — watch it live in the UI (http://202.131.110.56) meanwhile.

In [ ]:
import time
from clearml import Task
winner_id = None
for _ in range(40):   # up to ~10 min
    tasks = Task.get_tasks(project_name=PROJECT, task_name='ViT-Tiny finetune',
                           task_filter={'order_by': ['-last_update']})
    if tasks:
        t = tasks[0]; st = t.get_status()
        print('status:', st)
        if st in ('completed', 'failed', 'stopped', 'published'):
            break
    time.sleep(15)
print('latest training task:', tasks[0].id if tasks else None, '->', st if tasks else 'none')

## 5 · Hour 3 — compare runs & pick the winner

Rank your completed runs by validation accuracy and grab the best one's task id. *(In the UI you can also right-click a run → Clone → tweak a hyperparameter → Enqueue to `gpu-18gb`, then re-run this to compare.)*

In [ ]:
from clearml import Task
tasks = Task.get_tasks(project_name=PROJECT, task_name='ViT-Tiny finetune',
                       task_filter={'status': ['completed']})
def best_acc(t):
    for title, series in t.get_last_scalar_metrics().items():
        for name, stats in series.items():
            if 'accuracy' in f'{title}/{name}'.lower():
                return stats.get('max', stats.get('last'))
    return None
ranked = sorted(((t, best_acc(t)) for t in tasks),
                key=lambda kv: (kv[1] is not None, kv[1] or 0), reverse=True)
for t, acc in ranked:
    print(f'{acc!s:>8}  {t.name}  ({t.id})')
winner, winner_acc = ranked[0]
winner_id = winner.id
print('\nWINNER:', winner_id, '| accuracy', winner_acc)

## 6 · Hour 4 — publish the winner + run inference

Publish the winning model (tags it `winner`, makes it read-only), then run inference. Inference pulls the published model **by tag**, classifies a few images from the demo dataset, and reports the predictions grid (DEBUG SAMPLES) + a confusion matrix (PLOTS).

In [ ]:
# publish the winner (winner_id comes from Hour 3 above)
!python workshop/hour4_publish_and_infer.py publish --task-id {winner_id}

In [ ]:
# pull the published winner + classify sample images (no --images needed -> samples the demo set)
!python workshop/hour4_publish_and_infer.py infer

## ✅ Done

Open **http://202.131.110.56** → your project **`PlantVillage Workshop/<your-name>`**:
- **Hour 1 EDA task** → PLOTS (bar chart) + DEBUG SAMPLES (leaf grid)
- **ViT-Tiny finetune** → live loss/accuracy SCALARS + per-epoch models
- **Models** → your `winner` (published)
- **ViT-Tiny inference** → DEBUG SAMPLES (predictions) + PLOTS (confusion matrix)

## 🛠 Troubleshooting — common Colab issues

- **`API keys are EMPTY`** → you didn't paste the shared key (or `WORKSHOP_USER`) in the credentials cell. Fill them and re-run that cell.
- **`ModuleNotFoundError` / `!python` can't find clearml** → re-run the **Setup** cell (Colab reset the session). Do it after every reconnect.
- **Clone fails / Colab badge won't open** → the GitHub repo must be **public** (Colab pulls it anonymously). Ask the facilitator if it's private.
- **Hour 3 finds 0 runs** → Hour 2 hasn't finished yet. Run the **Wait for training** cell (or watch the task go *Completed* in the UI) first.
- **`Task fc… failed` in the UI** → open the task's **CONSOLE** tab for the real error. If it's a dataset/model `401 UNAUTHORIZED`, that's a server-side agent config issue — tell the facilitator.
- **`unauthenticated requests to the HF Hub`** warning → harmless (downloading the base image processor); ignore it.
- **Images show as an empty plot** → look in the **DEBUG SAMPLES** tab, not PLOTS (image grids are reported as PNGs there).
- **No GPU in Colab** → expected. Training runs on the server's H200 via the queue; Colab is just CPU for submitting + light EDA/inference.